**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Graph Signal Processing & GNNs

A flagship IEEE-SPS research area with almost no accessible teaching material: signals that live on **networks** — sensor grids, social graphs, molecules, power grids. Four sessions: the graph Laplacian gives graphs a Fourier transform, filters, and sampling theory — and message-passing GNNs drop out as learned graph filters. The oracle throughout: on a ring graph, everything must reduce to classical DSP.

## 1. Pre-requisites

- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S3 (eigendecomposition — this course is its victory lap).
- [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) (classical Fourier, for the reduction check).
- [CNN workshop](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) for Session 4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

def laplacian(A):
    return np.diag(A.sum(1)) - A

# our running graph: a random sensor network (geometric graph)
n_nodes = 80
pos = rng.random((n_nodes, 2))
D2 = ((pos[:, None] - pos[None]) ** 2).sum(-1)
A = ((D2 < 0.045) & (D2 > 0)).astype(float)
L = laplacian(A)
lam, U = np.linalg.eigh(L)                          # the graph's "frequencies" and "Fourier basis"

def draw(signal, title="", ax=None):
    if ax is None: fig, ax = plt.subplots(figsize=(3.6, 3.2))
    for i, j in zip(*np.nonzero(np.triu(A))):
        ax.plot(*zip(pos[i], pos[j]), "k-", linewidth=0.3, alpha=0.4)
    sc = ax.scatter(*pos.T, c=signal, s=45, cmap="coolwarm")
    ax.set_title(title, fontsize=9); ax.axis("off")
    return sc

---
### 🕐 Session 1 of 4 — *The Graph Laplacian & Graph Fourier Transform* (~40 min)
**Goal:** give any graph a frequency axis; verify it reduces to the DFT on a ring.
**Builds on:** [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S3. &nbsp; **Feeds into:** Session 2 (filtering on graphs).

---

## 2. Frequency Without Time

💡 **Intuition.** What does 'frequency' mean with no time axis? **Smoothness with respect to the edges.** The Laplacian quadratic form $x^T L x = \sum_{(i,j)\in E}(x_i - x_j)^2$ totals the disagreement across edges — so Laplacian eigenvectors, ordered by eigenvalue, are the graph's own harmonics: $\lambda \approx 0$ ⇒ smooth (neighbors agree), large $\lambda$ ⇒ oscillatory (neighbors alternate). The **graph Fourier transform** is just analysis in this eigenbasis: $\hat{x} = U^T x$ — [Hilbert-space](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) change of basis, with the graph choosing the basis.

In [ ]:

# YOUR CODE HERE


In [ ]:
# ORACLE: on a RING graph, the Laplacian eigenvalues must be the classical DFT frequencies

# YOUR CODE HERE


---
### 🕐 Session 2 of 4 — *Filtering on Graphs* (~40 min)
**Goal:** denoise a sensor field with a graph low-pass; make it local with polynomial filters.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (sampling).

---

## 3. Graph Filters

💡 **Intuition.** A graph filter scales each harmonic: $y = U h(\Lambda) U^T x$ — design $h(\lambda)$ exactly like a [filter response](./Filter_Design.ipynb), with $\lambda$ replacing $\omega$. The practical twist: eigendecomposition is $O(n^3)$, but a **polynomial** filter $h(L) = \sum_k c_k L^k$ needs only matrix-vector products — and $L^k x$ touches only $k$-hop neighbors, so polynomial order = *filter locality*. That locality is the seed GNNs grow from.

In [ ]:
# denoise a smooth temperature field over the sensor network
# local polynomial approximation of the same filter (Chebyshev-lite: least-squares fit)

# YOUR CODE HERE


---
### 🕐 Session 3 of 4 — *Sampling on Graphs* (~35 min)
**Goal:** which sensors can you afford to lose? Bandlimited recovery from a subset of nodes.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (GNNs).

---

## 4. Nyquist for Networks

💡 **Intuition.** If a graph signal is **bandlimited** — lives in the span of the first $K$ harmonics — then $K$ well-chosen node readings determine *all* $n$: solve the little least-squares system in the known coefficients ([Compressed Sensing's](./Compressed_Sensing.ipynb) logic, subspace version). 'Well-chosen' matters exactly like array geometry: sample nodes that make the harmonics distinguishable, not clustered clones of each other.

In [ ]:
# and the failure mode: measure fewer than K nodes → underdetermined

# YOUR CODE HERE


---
### 🕐 Session 4 of 4 — *Message Passing = Learned Graph Filters* (~40 min)
**Goal:** build a GCN from scratch; classify nodes; see it as Session 2 with trained coefficients.
**Builds on:** Session 3; [CNN workshop](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb).

---

## 5. GNNs, Demystified

💡 **Intuition.** A graph-convolution layer is: *average your neighbors (a fixed 1-hop low-pass $\hat{A} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$), then apply a learned linear map and a nonlinearity*. Stack $k$ layers ⇒ $k$-hop receptive field — precisely Session 2's polynomial filters with coefficients chosen by gradient descent. It's the [CNN story](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) (weight sharing + locality) generalized to irregular neighborhoods.

In [ ]:
# two-community node classification (a planted partition on top of geometry)

# YOUR CODE HERE


Ten labels classify eighty nodes because the graph *propagates* them — message passing is label smoothing through a learned low-pass. (Also visible here: stack too many layers and everything averages toward mush — *oversmoothing*, the graph version of over-aggressive low-pass filtering.)

## 6. Conclusion

The Laplacian gives every network a Fourier basis (reducing to the DFT on a ring — verified); filters are functions of $L$, made local by polynomials; bandlimited signals need only $K$ good sensors; and GNNs are those polynomial filters with learned coefficients. Classical DSP was the special case all along.

---
## Where next

- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) — the eigen-machinery, if it felt fast.
- [CNN workshop](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) — the regular-grid special case.
- [Statistical SP](./Statistical_Signal_Processing.ipynb) — stochastic graph signals are an open research door.